<h1>1. Import Required Libraries</h1>

In [1]:
import os
import nibabel as nib
import numpy as np
import pandas as pd
import glob
from scipy.ndimage import zoom

<h1>2. Configuration</h1>

<h3>2.1 Data Paths and Anatomical Priors</h3>

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

ROOT = "/workspace/data/raw"

TARGET_SPACING = np.array([1.0, 1.0, 1.0])

PRIORS = {
    "HEAD_NECK": {
        "x": 0.50,
        "y": 0.42,
        "z": 0.48
    },
    "FULL_BODY": {
        "x": 0.50,
        "y": 0.42,
        "z": 0.82
    }
}

<h3>2.2 Classify Scan Type</h3>

In [3]:
# ============================================================
# SCAN TYPE CLASSIFIER
# ============================================================

def classify_scan(size_z_mm):

    if size_z_mm >= 700:
        return "FULL_BODY"

    return "HEAD_NECK"

<h3>2.3  Get Cube Bounds</h3>

In [4]:
# ============================================================
# FIXED CUBE BOUNDS
# ============================================================

def get_cube_bounds(shape, center_xyz, cube_size):

    center_xyz = np.array(center_xyz).astype(int)

    start = center_xyz - cube_size // 2
    end = start + cube_size

    for d in range(3):

        if start[d] < 0:
            end[d] -= start[d]
            start[d] = 0

        if end[d] > shape[d]:

            shift = end[d] - shape[d]

            start[d] -= shift
            end[d] = shape[d]

        start[d] = max(start[d], 0)

    return start, end

<h1>3. Batch-wise Coverage Analysis</h1>
Due to storage limitations, the 782-patient HECKTOR 2026 training dataset was processed in batches rather than simultaneously. The coverage analysis was performed in batches of 200 patients, with the final batch containing the remaining patients (182). Patient-level coverage results were saved separately for each batch and subsequently combined for the final analysis across the complete dataset.

<h3>3.1 Compute Patient-Level GTV Coverage</h3>

In [5]:
# ============================================================
# COVERAGE ANALYSIS
# ============================================================

results = []

# Candidate crop sizes
CROP_SIZES = [160, 192, 224]

patients = sorted(os.listdir(ROOT))

for patient_no, p in enumerate(patients, start=1):

    patient_dir = os.path.join(ROOT, p)

    if not os.path.isdir(patient_dir):
        continue

    print(f"Processing {patient_no}: {p}")

    ct_path = os.path.join(
        patient_dir,
        f"{p}__CT.nii.gz"
    )

    mask_path = os.path.join(
        patient_dir,
        f"{p}.nii.gz"
    )

    if not (
        os.path.exists(ct_path)
        and os.path.exists(mask_path)
    ):
        print("Missing CT or mask")
        continue

    try:

        # ====================================================
        # LOAD
        # ====================================================

        ct_img = nib.load(ct_path)
        mask_img = nib.load(mask_path)

        ct = ct_img.get_fdata().astype(np.float32)
        mask = mask_img.get_fdata().astype(np.uint8)

        # ====================================================
        # SANITY CHECKS
        # ====================================================

        assert ct.shape == mask.shape

        ct_spacing = np.array(
            ct_img.header.get_zooms()[:3]
        )

        mask_spacing = np.array(
            mask_img.header.get_zooms()[:3]
        )

        assert np.allclose(
            ct_spacing,
            mask_spacing
        )

        # ====================================================
        # SCAN TYPE
        # ====================================================

        size_z_mm = (
            ct.shape[2]
            * ct_spacing[2]
        )

        scan_type = classify_scan(
            size_z_mm
        )

        # ====================================================
        # RESAMPLE TO 1 mm ISOTROPIC
        # ====================================================

        zoom_factor = (
            ct_spacing
            / TARGET_SPACING
        )

        ct_res = zoom(
            ct,
            zoom_factor,
            order=1
        ).astype(np.float32)

        mask_res = zoom(
            mask,
            zoom_factor,
            order=0
        ).astype(np.uint8)

        assert ct_res.shape == mask_res.shape

        # ====================================================
        # ANATOMICAL PRIOR CENTER
        # ====================================================

        sx, sy, sz = ct_res.shape

        center_x = int(
            PRIORS[scan_type]["x"] * sx
        )

        center_y = int(
            PRIORS[scan_type]["y"] * sy
        )

        center_z = int(
            PRIORS[scan_type]["z"] * sz
        )

        center = (
            center_x,
            center_y,
            center_z
        )

        # ====================================================
        # GTV MASKS
        # ====================================================

        # GTVp = label 1
        # GTVn = label 2

        gtvp = (mask_res == 1)
        gtvn = (mask_res == 2)

        gtvp_total = np.sum(gtvp)
        gtvn_total = np.sum(gtvn)

        # ====================================================
        # PATIENT RESULT
        # ====================================================

        patient_result = {
            "Patient": p,
            "Scan Type": scan_type,
        }

        # ====================================================
        # ANALYZE ALL CROP SIZES
        # ====================================================

        for cube_size_value in CROP_SIZES:

            cube_size = np.array(
                [cube_size_value] * 3
            )

            start, end = get_cube_bounds(
                ct_res.shape,
                center,
                cube_size
            )

            # ------------------------------------------------
            # Crop the ground-truth mask
            # ------------------------------------------------

            mask_crop = mask_res[
                start[0]:end[0],
                start[1]:end[1],
                start[2]:end[2]
            ]

            # ------------------------------------------------
            # Retained GTV voxels
            # ------------------------------------------------

            gtvp_crop = (mask_crop == 1)
            gtvn_crop = (mask_crop == 2)

            gtvp_retained = np.sum(gtvp_crop)
            gtvn_retained = np.sum(gtvn_crop)

            # ------------------------------------------------
            # Coverage
            # ------------------------------------------------

            gtvp_coverage = (
                100.0 * gtvp_retained / gtvp_total
                if gtvp_total > 0 else np.nan
            )

            gtvn_coverage = (
                100.0 * gtvn_retained / gtvn_total
                if gtvn_total > 0 else np.nan
            )

            # ------------------------------------------------
            # Store patient-level coverage
            # ------------------------------------------------

            prefix = f"{cube_size_value}"

            patient_result[
                f"GTVp Coverage (%) {prefix}"
            ] = gtvp_coverage

            patient_result[
                f"GTVn Coverage (%) {prefix}"
            ] = gtvn_coverage

        results.append(patient_result)

    except Exception as e:

        print(f"Failed on {p}")
        print(e)

Processing 1: MDA-277
Processing 2: MDA-278
Processing 3: MDA-279
Processing 4: MDA-280
Processing 5: MDA-281
Processing 6: MDA-282
Processing 7: MDA-283
Processing 8: MDA-284
Processing 9: MDA-285
Processing 10: MDA-286
Processing 11: MDA-287
Processing 12: MDA-288
Processing 13: MDA-289
Processing 14: MDA-290
Processing 15: MDA-291
Processing 16: MDA-292
Processing 17: MDA-293
Processing 18: MDA-294
Processing 19: MDA-295
Processing 20: MDA-296
Processing 21: MDA-297
Processing 22: MDA-298
Processing 23: MDA-299
Processing 24: MDA-300
Processing 25: MDA-301
Processing 26: MDA-302
Processing 27: MDA-303
Processing 28: MDA-304
Processing 29: MDA-305
Processing 30: MDA-306
Processing 31: MDA-307
Processing 32: MDA-308
Processing 33: MDA-309
Processing 34: MDA-310
Processing 35: MDA-311
Processing 36: MDA-312
Processing 37: MDA-313
Processing 38: MDA-314
Processing 39: MDA-315
Processing 40: MDA-316
Processing 41: MDA-317
Processing 42: MDA-318
Processing 43: MDA-319
Processing 44: MDA-3

<h3>3.2 Coverage Threshold Analysis</h3>
The patient-level coverage results are converted into a DataFrame and analyzed separately for GTVp and GTVn at each candidate crop size. For each tumor type, the analysis determines the number and percentage of cases in which at least 50%, 75%, 90%, 95%, 99%, or 100% of the ground-truth tumor voxels are retained within the crop. Cases with less than 50% coverage are also reported. Patients without a particular tumor annotation are excluded from the corresponding analysis.

In [6]:
# ============================================================
# CREATE PATIENT-LEVEL DATAFRAME
# ============================================================

df = pd.DataFrame(results)

print(f"Number of patients: {len(df)}")

# ============================================================
# COVERAGE THRESHOLD ANALYSIS
# ============================================================

THRESHOLDS = [100, 99, 95, 90, 75, 50]

threshold_results = []

for crop_size in CROP_SIZES:

    for tumor_type in ["GTVp", "GTVn"]:

        coverage_col = (
            f"{tumor_type} Coverage (%) {crop_size}"
        )

        valid = df[coverage_col].dropna()

        total = len(valid)

        for threshold in THRESHOLDS:

            count = np.sum(
                valid >= threshold
            )

            threshold_results.append({
                "Crop Size": f"{crop_size}³",
                "Tumor": tumor_type,
                "Coverage Threshold": (
                    "100%"
                    if threshold == 100
                    else f"≥{threshold}%"
                ),
                "Cases": count,
                "Total": total,
                "Percentage": (
                    100.0 * count / total
                    if total > 0 else np.nan
                )
            })

        # ----------------------------------------------------
        # Cases with <50% coverage
        # ----------------------------------------------------

        below_50 = np.sum(valid < 50)

        threshold_results.append({
            "Crop Size": f"{crop_size}³",
            "Tumor": tumor_type,
            "Coverage Threshold": "<50%",
            "Cases": below_50,
            "Total": total,
            "Percentage": (
                100.0 * below_50 / total
                if total > 0 else np.nan
            )
        })

threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df

Number of patients: 182


,Crop Size,Tumor,Coverage Threshold,Cases,Total,Percentage
0,160³,GTVp,100%,160,169,94.674556
1,160³,GTVp,≥99%,160,169,94.674556
2,160³,GTVp,≥95%,160,169,94.674556
3,160³,GTVp,≥90%,162,169,95.857988
4,160³,GTVp,≥75%,162,169,95.857988
5,160³,GTVp,≥50%,164,169,97.041420
6,160³,GTVp,<50%,5,169,2.958580
7,160³,GTVn,100%,134,164,81.707317
8,160³,GTVn,≥99%,138,164,84.146341
9,160³,GTVn,≥95%,145,164,88.414634


<h3>3.3 Save Batch-Level Results</h3>
The patient-level coverage results for the current batch are saved as a CSV file. Each batch is assigned a unique identifier in BATCH_ID, allowing the results from all batches to be combined later for the final analysis across the complete HECKTOR 2026 training cohort.

In [7]:
# ============================================================
# SAVE PATIENT-LEVEL RESULTS
# ============================================================

BATCH_ID = "04"

df = pd.DataFrame(results)

output_file = (
    f"crop_coverage_patient_level_batch_{BATCH_ID}.csv"
)

df.to_csv(
    output_file,
    index=False
)

print()
print(
    f"Saved patient-level results to: {output_file}"
)


Saved patient-level results to: crop_coverage_patient_level_batch_04.csv


<h3>3.4 Generate and Save Batch-Level Coverage Summary</h3>
This section summarizes the coverage results for the current batch across all candidate crop sizes and for both GTVp and GTVn. For each tumor type, the number and percentage of cases meeting the specified cumulative coverage thresholds (≥50%, ≥75%, ≥90%, ≥95%, ≥99%, and 100%) are reported. Cases with less than 50% coverage are reported separately. Patients without the corresponding tumor annotation are excluded from the denominator. The resulting batch-level summary is saved as a CSV file for subsequent aggregation with the results from the other batches.

In [8]:
# ============================================================
# COVERAGE THRESHOLD SUMMARY
# ============================================================

THRESHOLDS = [100, 99, 95, 90, 75, 50]

summary = []

for cube_size in CROP_SIZES:

    for tumor_type in ["GTVp", "GTVn"]:

        coverage_col = (
            f"{tumor_type} Coverage (%) {cube_size}"
        )

        valid = df[coverage_col].dropna()

        total = len(valid)

        # ----------------------------------------------------
        # Cumulative coverage thresholds
        # ----------------------------------------------------

        for threshold in THRESHOLDS:

            count = np.sum(
                valid >= threshold
            )

            summary.append({
                "Crop Size": f"{cube_size}³",
                "Tumor": tumor_type,
                "Coverage Threshold": (
                    "100%"
                    if threshold == 100
                    else f"≥{threshold}%"
                ),
                "Cases": count,
                "Total": total,
                "Percentage": (
                    100.0 * count / total
                    if total > 0 else np.nan
                )
            })

        # ----------------------------------------------------
        # Less than 50% coverage
        # ----------------------------------------------------

        count = np.sum(valid < 50)

        summary.append({
            "Crop Size": f"{cube_size}³",
            "Tumor": tumor_type,
            "Coverage Threshold": "<50%",
            "Cases": count,
            "Total": total,
            "Percentage": (
                100.0 * count / total
                if total > 0 else np.nan
            )
        })


summary_df = pd.DataFrame(summary)

print("\n========================================")
print("CROP COVERAGE THRESHOLD SUMMARY")
print("========================================")

print(
    summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

summary_output_file = (
    f"crop_coverage_summary_batch_{BATCH_ID}.csv"
)

summary_df.to_csv(
    summary_output_file,
    index=False
)

print()
print(
    f"Saved summary to: {summary_output_file}"
)


CROP COVERAGE THRESHOLD SUMMARY
Crop Size Tumor Coverage Threshold  Cases  Total  Percentage
     160³  GTVp               100%    160    169       94.67
     160³  GTVp               ≥99%    160    169       94.67
     160³  GTVp               ≥95%    160    169       94.67
     160³  GTVp               ≥90%    162    169       95.86
     160³  GTVp               ≥75%    162    169       95.86
     160³  GTVp               ≥50%    164    169       97.04
     160³  GTVp               <50%      5    169        2.96
     160³  GTVn               100%    134    164       81.71
     160³  GTVn               ≥99%    138    164       84.15
     160³  GTVn               ≥95%    145    164       88.41
     160³  GTVn               ≥90%    151    164       92.07
     160³  GTVn               ≥75%    155    164       94.51
     160³  GTVn               ≥50%    161    164       98.17
     160³  GTVn               <50%      3    164        1.83
     192³  GTVp               100%    163    169    

Verify the number of patients with and without GTVp/GTVn annotations.

In [9]:
print("Total patients:", len(df))

print(
    "Patients with GTVp:",
    df["GTVp Coverage (%) 160"].notna().sum()
)

print(
    "Patients with GTVn:",
    df["GTVn Coverage (%) 160"].notna().sum()
)

print(
    "Patients without GTVp:",
    df["GTVp Coverage (%) 160"].isna().sum()
)

print(
    "Patients without GTVn:",
    df["GTVn Coverage (%) 160"].isna().sum()
)

Total patients: 182
Patients with GTVp: 169
Patients with GTVn: 164
Patients without GTVp: 13
Patients without GTVn: 18


<h1>4. Aggregate Results Across All Batches</h1>

<h3>4.1 Combine Patient-Level Results</h3>
Identify and combine the patient-level results from all four batches.
Verify that all four batch files are present and that the combined dataset contains 782 patients.

In [11]:
files = sorted(
    glob.glob("crop_coverage_patient_level_batch_*.csv")
)

print("Batch files found:", len(files))

for f in files:
    print(" -", f)

assert len(files) == 4, (
    f"Expected 4 batch files, but found {len(files)}."
)

df_all = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True
)

print()
print("Number of patients:", len(df_all))

assert len(df_all) == 782, (
    f"Expected 782 patients, but found {len(df_all)}."
)

Batch files found: 4
 - crop_coverage_patient_level_batch_01.csv
 - crop_coverage_patient_level_batch_02.csv
 - crop_coverage_patient_level_batch_03.csv
 - crop_coverage_patient_level_batch_04.csv

Number of patients: 782


<h3>4.2 Final Coverage Threshold Analysis</h3>
The patient-level results from all four batches are combined to quantify GTVp and GTVn coverage across the complete 782-patient training cohort. For each candidate crop size (160³, 192³, and 224³), the analysis reports the number and percentage of cases meeting each cumulative coverage threshold. Patients without the corresponding tumor annotation are excluded from the denominator. Duplicate patient identifiers are also checked to ensure that each patient contributes only once to the final analysis.

In [12]:
# ============================================================
# FINAL GTV COVERAGE ANALYSIS — ALL 782 PATIENTS
# ============================================================

CROP_SIZES = [160, 192, 224]

# Cumulative thresholds
THRESHOLDS = [100, 99, 95, 90, 75, 50]

# ------------------------------------------------------------
# Verify the combined dataset
# ------------------------------------------------------------

EXPECTED_PATIENTS = 782

assert len(df_all) == EXPECTED_PATIENTS, (
    f"Expected {EXPECTED_PATIENTS} patients, "
    f"but found {len(df_all)}."
)

print(f"Verified patient count: {len(df_all)}")

# Check for duplicate patients
duplicate_count = df_all["Patient"].duplicated().sum()

assert duplicate_count == 0, (
    f"Found {duplicate_count} duplicate patient IDs."
)

print("No duplicate patients found.")

# ------------------------------------------------------------
# Calculate threshold statistics
# ------------------------------------------------------------

final_results = []

for crop_size in CROP_SIZES:

    for tumor_type in ["GTVp", "GTVn"]:

        coverage_col = (
            f"{tumor_type} Coverage (%) {crop_size}"
        )

        valid = df_all[coverage_col].dropna()

        total = len(valid)

        # ----------------------------------------------------
        # Cumulative thresholds
        # ----------------------------------------------------

        for threshold in THRESHOLDS:

            count = np.sum(
                valid >= threshold
            )

            percentage = (
                100.0 * count / total
                if total > 0 else np.nan
            )

            threshold_label = (
                "100%"
                if threshold == 100
                else f"≥{threshold}%"
            )

            final_results.append({
                "Crop Size": f"{crop_size}³",
                "Tumor": tumor_type,
                "Coverage": threshold_label,
                "Cases": int(count),
                "Total": int(total),
                "Percentage (%)": percentage
            })

        # ----------------------------------------------------
        # <50% coverage
        # ----------------------------------------------------

        count = np.sum(valid < 50)

        percentage = (
            100.0 * count / total
            if total > 0 else np.nan
        )

        final_results.append({
            "Crop Size": f"{crop_size}³",
            "Tumor": tumor_type,
            "Coverage": "<50%",
            "Cases": int(count),
            "Total": int(total),
            "Percentage (%)": percentage
        })


final_df = pd.DataFrame(final_results)

# ============================================================
# DISPLAY RESULTS
# ============================================================

print()
print("=" * 70)
print("FINAL GTV COVERAGE ANALYSIS — 782 PATIENTS")
print("=" * 70)

print(
    final_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

# ============================================================
# SAVE FINAL RESULTS
# ============================================================

final_output_file = (
    "crop_coverage_final_782_patients.csv"
)

final_df.to_csv(
    final_output_file,
    index=False
)

print()
print(
    f"Saved final analysis to: {final_output_file}"
)

Verified patient count: 782
No duplicate patients found.

FINAL GTV COVERAGE ANALYSIS — 782 PATIENTS
Crop Size Tumor Coverage  Cases  Total  Percentage (%)
     160³  GTVp     100%    674    751           89.75
     160³  GTVp     ≥99%    683    751           90.95
     160³  GTVp     ≥95%    696    751           92.68
     160³  GTVp     ≥90%    702    751           93.48
     160³  GTVp     ≥75%    717    751           95.47
     160³  GTVp     ≥50%    732    751           97.47
     160³  GTVp     <50%     19    751            2.53
     160³  GTVn     100%    552    695           79.42
     160³  GTVn     ≥99%    572    695           82.30
     160³  GTVn     ≥95%    602    695           86.62
     160³  GTVn     ≥90%    633    695           91.08
     160³  GTVn     ≥75%    657    695           94.53
     160³  GTVn     ≥50%    676    695           97.27
     160³  GTVn     <50%     19    695            2.73
     192³  GTVp     100%    726    751           96.67
     192³  GTVp    

<h3>4.3 Paper-Friendly Coverage Table</h3>
Generate a compact table summarizing GTV coverage across crop sizes and coverage thresholds for reporting.

In [13]:
# ============================================================
# PAPER-FRIENDLY COVERAGE TABLE
# ============================================================

paper_rows = []

for crop_size in CROP_SIZES:

    for threshold in THRESHOLDS:

        if threshold == 100:
            label = "100%"
        else:
            label = f"≥{threshold}%"

        row = {
            "Crop Size": f"{crop_size}³",
            "GTV Coverage": label
        }

        for tumor_type in ["GTVp", "GTVn"]:

            coverage_col = (
                f"{tumor_type} Coverage (%) {crop_size}"
            )

            valid = df_all[coverage_col].dropna()

            count = np.sum(valid >= threshold)
            total = len(valid)

            row[tumor_type] = (
                f"{count}/{total} "
                f"({100.0 * count / total:.1f}%)"
                if total > 0
                else "N/A"
            )

        paper_rows.append(row)

    # --------------------------------------------------------
    # <50%
    # --------------------------------------------------------

    row = {
        "Crop Size": f"{crop_size}³",
        "GTV Coverage": "<50%"
    }

    for tumor_type in ["GTVp", "GTVn"]:

        coverage_col = (
            f"{tumor_type} Coverage (%) {crop_size}"
        )

        valid = df_all[coverage_col].dropna()

        count = np.sum(valid < 50)
        total = len(valid)

        row[tumor_type] = (
            f"{count}/{total} "
            f"({100.0 * count / total:.1f}%)"
            if total > 0
            else "N/A"
        )

    paper_rows.append(row)


paper_df = pd.DataFrame(paper_rows)

print()
print("=" * 70)
print("PAPER-FRIENDLY GTV COVERAGE TABLE")
print("=" * 70)

print(
    paper_df.to_string(index=False)
)

paper_df.to_csv(
    "crop_coverage_paper_table.csv",
    index=False
)


PAPER-FRIENDLY GTV COVERAGE TABLE
Crop Size GTV Coverage            GTVp            GTVn
     160³         100% 674/751 (89.7%) 552/695 (79.4%)
     160³         ≥99% 683/751 (90.9%) 572/695 (82.3%)
     160³         ≥95% 696/751 (92.7%) 602/695 (86.6%)
     160³         ≥90% 702/751 (93.5%) 633/695 (91.1%)
     160³         ≥75% 717/751 (95.5%) 657/695 (94.5%)
     160³         ≥50% 732/751 (97.5%) 676/695 (97.3%)
     160³         <50%   19/751 (2.5%)   19/695 (2.7%)
     192³         100% 726/751 (96.7%) 639/695 (91.9%)
     192³         ≥99% 729/751 (97.1%) 651/695 (93.7%)
     192³         ≥95% 734/751 (97.7%) 663/695 (95.4%)
     192³         ≥90% 736/751 (98.0%) 671/695 (96.5%)
     192³         ≥75% 741/751 (98.7%) 682/695 (98.1%)
     192³         ≥50% 747/751 (99.5%) 690/695 (99.3%)
     192³         <50%    4/751 (0.5%)    5/695 (0.7%)
     224³         100% 744/751 (99.1%) 674/695 (97.0%)
     224³         ≥99% 745/751 (99.2%) 676/695 (97.3%)
     224³         ≥95% 747/751